In [1]:
import Tensor as t
import numpy as np
import matplotlib.pyplot as plt
import Operations as o
import Compile as c
from sklearn.datasets import fetch_openml

We will stick to the row major order to align with numpy. This means our "dense" layers will be

$$\bold{Y} = \bold{X}\bold{W} + \bold{b}$$

Where $\bold{X}$ is the row vector in question. Might implement a technique called batching in the future.

In [2]:
# Fetch the MNIST dataset (this might take a minute to download)
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

# Split into features (images) and labels
X, y = mnist["data"], mnist["target"]

print(f"Dataset shape: {X.shape}")

Dataset shape: (70000, 784)


In [3]:
def encode(val):
    z = np.zeros(10,dtype=np.float64)
    z[int(val)] += 1.0
    return z

y_cleaned = np.array([np.array([encode(k)]) for k in y]) #shape = (70_000,1,10)

x_cleaned = X.astype(np.float64) / 255
x_cleaned = x_cleaned.reshape((70_000,1,784))


In [4]:
x_train, x_test = np.split(x_cleaned,[50_000])
y_train, y_test = np.split(y_cleaned,[50_000])

In [5]:
x_train = np.sum(x_train,axis=(1))
y_train = np.sum(y_train,axis=(1))

In [6]:
print(y_train.shape)

(50000, 10)


In [7]:
#current architecture: Dense(784 15) sAct softmax Dense(15 10) sAct softmax (done!)
#paramaters
w_1 = np.random.random_sample(size=(784, 15))
b_1 = np.random.random_sample(size=(1,15))

w_2 = np.random.random_sample(size=(15,10))
b_2 = np.random.random_sample(size=(1,10))


In [8]:
#the input change this in order to change the input
x_value = t.TensorNode(x_cleaned[0],is_param=False)
L_1 = x_value @ t.TensorNode(w_1) + t.TensorNode(b_1)
L_1 = (o.sAct(L_1))
L_1 = (L_1)/(o.sum(L_1,axis=(1),keepDim=True)+1e-7)

L_2 = L_1 @ t.TensorNode(w_2) + t.TensorNode(b_2)
L_2 = o.sAct(L_2)
L_2 = (L_2)/(o.sum(L_2,axis=(1),keepDim = True) + 1e-7)

#model variable -- use this to peek at output
model = L_2

In [9]:
print((model.data))

[[0.08126189 0.11754752 0.10393509 0.1116604  0.12640261 0.12263588
  0.07229046 0.09264025 0.09290856 0.07871733]]


In [10]:
y_value = t.TensorNode(y_cleaned[0],is_param=False)

In [11]:
loss = o.norm_squared(model - y_value)
comp = c.Pipeline(loss.compile())

In [12]:
print(loss.data)
for j in range(50_000):
    x_value.data = x_cleaned[j]
    y_value.data = y_cleaned[j]
    comp.update_input()
    comp.train()
    comp.update(3)

print(loss.data)

0.8580734251526372
8.433209809791028e-05


In [13]:
#testing out new method
x_value.data = x_train
y_value.data = y_train
print(loss.data)
comp.update_input()
comp.train()
comp.update(0.3)
comp.update_input()
print(loss.data)

8.433209809791028e-05


ValueError: non-broadcastable output operand with shape (1,10) doesn't match the broadcast shape (50000,10)

In [14]:
total = 0
for j in range(50_000):
    x_value.data = x_cleaned[j]
    y_value.data = y_cleaned[j]
    comp.update_input()
    total += loss.data

print(total)

7514.766650008508


63108.25813250778
(15, 70000) (70000, 10) (10, 15)
(784, 70000) (70000, 15) (15, 784)
new lr 5.0
64634.11621406724

WE DID IT. Below is the accuracy:

In [15]:
correct = 0
for j in range(50_000):
    x_value.data = x_cleaned[j]
    comp.update_input()
    if(str(np.argmax(model.data)) == y[j]):
        correct+=1

print("Raw correct: " + str(correct))
print("accruacy " + str(correct/50_000))

print("test dataset:")
correct = 0
for j in range(50_001,70_000):
    x_value.data = x_cleaned[j]
    comp.update_input()
    if(str(np.argmax(L_2.data)) == y[j]):
        correct+=1

print("Raw correct: " + str(correct))
print("accruacy " + str(correct/20_000))


Raw correct: 45369
accruacy 0.90738
test dataset:
Raw correct: 18257
accruacy 0.91285


Raw correct: 5101
accruacy 0.10202
test dataset:
Raw correct: 2039
accruacy 0.10195

Check out data.npz to import paramaters